# Spotify — `songs_with_attributes_and_lyrics.csv` Preprocessing

Loads the raw Spotify dataset from `data/raw/`, applies all cleaning and deduplication steps for a **lyrics-based unsupervised recommendation system**, and saves the result to `data/processed/spotify_clean.csv`.

**Constraints:** no NLP libraries · no stemming/lemmatisation · no stopword removal · preserve lyric structure (line breaks).

In [39]:
# Bring in the tools we need:
# re        -> for finding and replacing patterns in text (like removing punctuation)
# hashlib   -> for creating a unique fingerprint from lyrics text
# Path      -> makes file/folder paths easy to work with on any OS
# pandas    -> the main library for loading and working with spreadsheet-like data
import re
import hashlib
from pathlib import Path
import pandas as pd

In [40]:
# ── Paths — locate the application root (`approot`) ───────────────────────
# Simple, predictable rules:
# - If running inside the project's .venv, go up one level and use `approot` there.
# - Otherwise search upward up to a few levels for a folder named `approot`.
# - If not found, fall back to the parent folder (best-effort).
from pathlib import Path

cwd = Path.cwd()
# If notebook is executed from inside .venv, move up to the repo folder
if cwd.name == '.venv':
    base = cwd.parent
else:
    base = cwd

# Prefer an explicit `approot` directory if present
APP_ROOT = None
if (base / 'approot').exists():
    APP_ROOT = (base / 'approot').resolve()
else:
    p = base
    for _ in range(6):
        if (p / 'approot').exists():
            APP_ROOT = (p / 'approot').resolve()
            break
        if p.parent == p:
            break
        p = p.parent

# final fallback: use base (project root candidate)
if APP_ROOT is None:
    APP_ROOT = base.resolve()

RAW_CSV = APP_ROOT / 'data' / 'raw' / 'songs_with_attributes_and_lyrics.csv'
PROCESSED_DIR = APP_ROOT / 'data' / 'processed'
OUT_CSV = PROCESSED_DIR / 'spotify_clean.csv'

print('app root:   ', APP_ROOT)
print('raw exists?  ', RAW_CSV.exists())
print('output will be:', OUT_CSV)

app root:    C:\Users\Aman Tewari\Documents\aaa-college\sem6\SLA-pvt\approot
raw exists?   True
output will be: C:\Users\Aman Tewari\Documents\aaa-college\sem6\SLA-pvt\approot\data\processed\spotify_clean.csv


In [41]:
# Reusable helper functions
# These are small tools we'll call repeatedly during cleaning.

def ascii_ratio(text: str) -> float:
    # Return fraction of characters that are ASCII (common in English text).
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return sum(ord(c) < 128 for c in text) / len(text)


def normalize_meta(s: str) -> str:
    # Clean a title or artist string to a simple canonical form.
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r"\([^)]*(live|remaster(ed)?).*?\)", '', s, flags=re.IGNORECASE)
    s = re.sub(r"[\-\u2013\u2014]\s*live\s*$", '', s, flags=re.IGNORECASE)
    s = re.sub(r"\([^)]*\)", '', s)
    s = re.sub(r"[^\w\s]", ' ', s)
    return re.sub(r"\s+", ' ', s).strip()


def normalize_lyrics(text: str) -> str:
    # Lowercase, remove most punctuation but keep apostrophes, and normalize spaces per line.
    if not isinstance(text, str):
        return ''
    t = text.lower()
    t = re.sub(r"[^\w\s']", ' ', t)
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in t.splitlines()]
    return '\n'.join(lines)


def md5_hash(s: str) -> str:
    # Fast fingerprint for deduplication
    return hashlib.md5(s.encode('utf-8')).hexdigest()

In [42]:
# ── Step 1 · Load the raw data
# Read the CSV file into a pandas DataFrame (think of it like an Excel table).
# We check the file exists first so we get a helpful error message instead of a cryptic crash.
if not RAW_CSV.exists():
    raise FileNotFoundError(
        f'Raw CSV not found at {RAW_CSV}\n'
        'Place songs_with_attributes_and_lyrics.csv inside data/raw/ and re-run.'
    )

# low_memory=False tells pandas to read the whole file before deciding column types
# — avoids misleading "mixed types" warnings on large files.
df = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Loaded  : {len(df):,} rows  |  columns: {list(df.columns)}')

Loaded  : 955,320 rows  |  columns: ['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics']


In [43]:
# ── Step 2 · Drop columns we do not need
# Keep only id, name, artists, lyrics and rename for clarity.
REQUIRED = ['id', 'name', 'artists', 'lyrics']
missing  = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns in CSV: {missing}')

df = df[REQUIRED].copy()
df = df.rename(columns={'name': 'title', 'artists': 'artist'})
print(f'After column selection  : {len(df):,} rows')

After column selection  : 955,320 rows


In [44]:
# ── Step 3 · Remove songs with missing or very short lyrics
# Remove empty lyrics, and songs with fewer than 30 words
df = df[df['lyrics'].notna()].copy()
df['_word_count'] = df['lyrics'].astype(str).str.split().str.len()
df = df[df['_word_count'] >= 30].copy()
print(f'After null/short filter : {len(df):,} rows')

After null/short filter : 947,404 rows


In [45]:
# ── Step 4 · Keep English-language songs only
# Use ASCII-ratio heuristic: keep songs where >=90% of characters are ASCII
df['_ascii'] = df['lyrics'].astype(str).map(ascii_ratio)
df = df[df['_ascii'] >= 0.90].copy()
print(f'After English filter    : {len(df):,} rows')

After English filter    : 889,697 rows


In [46]:
# ── Step 5 · Clean up song titles and artist names
# Create a stricter normalized title for deduplication while keeping artist cleaning

def normalize_title(s: str) -> str:
    # 1) lowercase
    # 2) remove leading/trailing non-alphanumeric characters
    # 3) replace non-alphanumeric (except spaces) with a single space
    # 4) collapse multiple spaces
    # 5) strip
    if not isinstance(s, str):
        return ''
    t = s.lower()
    t = re.sub(r'^[^A-Za-z0-9]+|[^A-Za-z0-9]+$', '', t)   # trim non-alphanum at edges
    t = re.sub(r'[^A-Za-z0-9 ]+', ' ', t)                 # replace non-alphanum (except space) with space
    t = re.sub(r'\s+', ' ', t).strip()                   # collapse spaces and trim
    return t

# Apply title normalization and keep artist normalized using existing helper
df['normalized_title'] = df['title'].astype(str).map(normalize_title)
df['normalized_artist'] = df['artist'].astype(str).map(normalize_meta)

# Show a small sample of before -> after so we can inspect the changes
print('Sample original -> normalized (first 10 rows):')
print(df[['title', 'normalized_title']].head(10).to_string(index=False))

# Drop rows where normalized title is invalid per rules:
# - empty, length < 2, or contains only digits
before_titles = len(df)
mask_invalid_title = (
    df['normalized_title'].isna()
    | (df['normalized_title'].str.len() < 2)
    | (df['normalized_title'].str.strip() == '')
    | (df['normalized_title'].str.fullmatch(r'\d+'))
)
# Report and remove invalids
num_invalid_titles = int(mask_invalid_title.sum())
if num_invalid_titles > 0:
    print(f'Removing {num_invalid_titles} rows due to invalid normalized titles')
df = df[~mask_invalid_title].copy()


Sample original -> normalized (first 10 rows):
                        title            normalized_title
                            !                            
                           !!                            
               !!De Repente!!                  de repente
               !!De Repente!!                  de repente
          !!Noble Stabbings!!             noble stabbings
               !I'll Be Back!                i ll be back
                       !Lost!                        lost
    !Que Vida! - Mono Version       que vida mono version
!Viva el Mal Viva el Capital! viva el mal viva el capital
                   !liaF cipE                   liaf cipe
Removing 1770 rows due to invalid normalized titles


In [47]:
# ── Step 6 · Remove perfectly identical rows
before = len(df)
df = df.drop_duplicates().copy()
print(f'Removed exact duplicates        : {before - len(df):,}  →  {len(df):,} remain')

Removed exact duplicates        : 0  →  887,927 remain


In [48]:
# ── Step 7 · Remove duplicates that are the same song by a different name
# Use composite key of normalized_title + normalized_artist
# Ensure the normalized fields exist (they were created in the title-cleaning step)
if 'normalized_title' not in df.columns:
    df['normalized_title'] = df['title'].astype(str).map(normalize_title)
if 'normalized_artist' not in df.columns:
    df['normalized_artist'] = df['artist'].astype(str).map(normalize_meta)

# Build composite key and drop duplicates keeping the first occurrence
df['_comp_key'] = df['normalized_title'].fillna('') + '_' + df['normalized_artist'].fillna('')
before = len(df)
df = df.drop_duplicates(subset=['_comp_key']).copy()
print(f'Removed composite-key duplicates: {before - len(df):,}  →  {len(df):,} remain')

Removed composite-key duplicates: 109,876  →  778,051 remain


In [49]:
# ── Step 8 · Clean lyrics text and remove songs with identical lyrics
# Normalize lyrics (preserve line breaks) and use MD5 to fingerprint
df['_lyrics_norm'] = df['lyrics'].astype(str).map(normalize_lyrics)
df['_lyrics_hash'] = df['_lyrics_norm'].map(md5_hash)
before = len(df)
df = df.drop_duplicates(subset=['_lyrics_hash']).copy()
print(f'Removed duplicate lyrics (hash) : {before - len(df):,}  →  {len(df):,} remain')

Removed duplicate lyrics (hash) : 28,893  →  749,158 remain


In [50]:
# ── Step 9 · Remove songs that are unusually short or absurdly long
wc      = df['_word_count']
q1, q3  = wc.quantile(0.25), wc.quantile(0.75)
iqr     = q3 - q1
lo      = max(1, int(q1 - 1.5 * iqr))
hi      = int(q3 + 1.5 * iqr)
before  = len(df)
df = df[(df['_word_count'] >= lo) & (df['_word_count'] <= hi)].copy()
print(f'IQR word-count bounds           : [{lo}, {hi}]')
print(f'Removed outliers                : {before - len(df):,}  →  {len(df):,} remain')

IQR word-count bounds           : [1, 568]
Removed outliers                : 44,815  →  704,343 remain


In [51]:
# ── Step 10 · Save the clean dataset
final_df = df[['id', 'title', 'artist', 'lyrics']].copy()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUT_CSV, index=False)
print(f'\nSaved → {OUT_CSV}')
print(f'Final row count : {len(final_df):,}')
final_df.head(5)


Saved → C:\Users\Aman Tewari\Documents\aaa-college\sem6\SLA-pvt\approot\data\processed\spotify_clean.csv
Final row count : 704,343


,id,title,artist,lyrics
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,['Rosendo'],Continuamente se extraña la gente si no puede ...
5,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,Dillinger Four,You like to stand on the other side\n Point an...
6,0fROT4kK5oTm8xO8PX6EJF,!I'll Be Back!,Rilès,"It's been a while, shit I missed the rehab, ps..."
7,1xBFhv5faebv3mmwxx7DnS,!Lost!,Rilès,I would like to give you all my time\n I would...
8,0gNNToCW3qjabgTyBSjt3H,!Que Vida! - Mono Version,['Love'],With pictures and words\n Is this communicatin...
